In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from catboost import CatBoostClassifier

In [ ]:
df = pd.read_csv("Data/filtered-dataset.csv")
df.head()

C:\Users\rashe\AppData\Local\Temp\ipykernel_8560\3962052459.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Data/filtered-dataset.csv")


,datetime,event,temp,airTemp,humidity,accel,rumination,ageInDays,aveHerdTemp,date,time,id,THI,age_category,rumination_per_day,farm_id,label
0,2025-03-09 23:56:20,NaN,39.0,13.8,58.7,0.20,0,147,38.56,2025-03-09,23:56:20,585,57.09,Calf,60,74,0
1,2025-03-10 00:11:20,NaN,39.3,13.8,58.7,0.20,0,147,38.05,2025-03-10,00:11:20,585,57.09,Calf,135,74,0
2,2025-03-10 00:26:20,NaN,36.6,13.9,63.2,0.22,0,147,38.05,2025-03-10,00:26:20,585,57.20,Calf,135,74,0
3,2025-03-10 00:41:20,NaN,38.2,14.1,60.4,0.17,0,147,38.05,2025-03-10,00:41:20,585,57.50,Calf,135,74,0
4,2025-03-10 00:56:20,NaN,39.0,13.9,58.7,0.19,0,147,38.05,2025-03-10,00:56:20,585,57.23,Calf,135,74,0


In [ ]:

df = df.drop(columns=["event"])

In [17]:
df.head()

,datetime,temp,airTemp,humidity,accel,rumination,ageInDays,aveHerdTemp,id,THI,rumination_per_day
0,2025-03-09 23:56:20,39.0,13.8,58.7,0.20,0,147,38.56,585,57.09,60
1,2025-03-10 00:11:20,39.3,13.8,58.7,0.20,0,147,38.05,585,57.09,135
2,2025-03-10 00:26:20,36.6,13.9,63.2,0.22,0,147,38.05,585,57.20,135
3,2025-03-10 00:41:20,38.2,14.1,60.4,0.17,0,147,38.05,585,57.50,135
4,2025-03-10 00:56:20,39.0,13.9,58.7,0.19,0,147,38.05,585,57.23,135


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 881359 entries, 0 to 881358
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   datetime            881359 non-null  object 
 1   temp                881359 non-null  float64
 2   airTemp             881359 non-null  float64
 3   humidity            881359 non-null  float64
 4   accel               881359 non-null  float64
 5   rumination          881359 non-null  int64  
 6   ageInDays           881359 non-null  int64  
 7   aveHerdTemp         881359 non-null  float64
 8   id                  881359 non-null  int64  
 9   THI                 881359 non-null  float64
 10  rumination_per_day  881359 non-null  int64  
dtypes: float64(6), int64(4), object(1)
memory usage: 74.0+ MB


In [24]:
def add_fever_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 'label' column: 0 = healthy, 1 = sick
    Based on 6-hour rolling window:
      - Sick (1) if: at least 3 temp readings > 40.0 OR at least 1 reading > 41.0
    """
    if "temp" not in df.columns:
        raise ValueError("Expected 'temp' column in the data.")
    
    rows = []
    for calf_id, group in df.groupby("id"):
        group = group.sort_values("datetime").copy()

        # Ensure datetime format
        group["datetime"] = pd.to_datetime(group["datetime"], errors="coerce")

        # Set datetime index (required for time-based rolling)
        group = group.set_index("datetime").sort_index()

        temps = group["temp"]
        
        # 6-hour rolling window
        gt_40_0 = (temps > 40.0).rolling("6H", min_periods=1).sum()
        gt_41_0 = (temps > 41.0).rolling("6H", min_periods=1).sum()
        
        # Sick rule
        fever_flag = (gt_40_0 >= 3) | (gt_41_0 >= 1)
        
        group["label"] = fever_flag.astype(int)

        group = group.reset_index()
        rows.append(group)
    
    return pd.concat(rows, ignore_index=True)


# Add the label column
df = add_fever_labels(df)

# Check the distribution
print(df["label"].value_counts())
print(f"\nPercentage sick: {df['label'].mean() * 100:.2f}%")

C:\Users\rashe\AppData\Local\Temp\ipykernel_8560\2830538852.py:23: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  gt_40_0 = (temps > 40.0).rolling("6H", min_periods=1).sum()
C:\Users\rashe\AppData\Local\Temp\ipykernel_8560\2830538852.py:24: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  gt_41_0 = (temps > 41.0).rolling("6H", min_periods=1).sum()
C:\Users\rashe\AppData\Local\Temp\ipykernel_8560\2830538852.py:23: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  gt_40_0 = (temps > 40.0).rolling("6H", min_periods=1).sum()
C:\Users\rashe\AppData\Local\Temp\ipykernel_8560\2830538852.py:24: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  gt_41_0 = (temps > 41.0).rolling("6H", min_periods=1).sum()
C:\Users\rashe\AppData\Local\Temp\ipykernel_8560\2830538852.py:23: FutureWarning: 'H' is dep

label
0    708380
1    172979
Name: count, dtype: int64

Percentage sick: 19.63%


In [ ]:
def add_rolling_features(df: pd.DataFrame, window_hours: int = 6) -> pd.DataFrame:
    """
    Add 6-hour rolling mean/std for numeric features per calf (id).
    """
    rows = []
    window_str = f"{window_hours}H"

    for calf_id, group in df.groupby("id"):
        group = group.sort_values("datetime").copy()
        group["datetime"] = pd.to_datetime(group["datetime"], errors="coerce")
        group = group.set_index("datetime").sort_index()

        numeric_cols = group.select_dtypes(include=[np.number]).columns.tolist()
        base_numeric = [c for c in numeric_cols if c != 'label']

        for col in base_numeric:
            roll = group[col].rolling(window_str, min_periods=1)
            group[f"{col}_6h_mean"] = roll.mean().values
            group[f"{col}_6h_std"] = roll.std(ddof=0).values

        group = group.reset_index()
        rows.append(group)

    return pd.concat(rows, ignore_index=True)

df = add_rolling_features(df, window_hours=6)  

feature_cols = [c for c in df.columns 
                if c not in ['label', 'id', 'datetime'] 
                and pd.api.types.is_numeric_dtype(df[c])]


In [31]:
print(f"Total features: {len(feature_cols)}")
print(f"Sample features: {feature_cols[:9]}")

Total features: 29
Sample features: ['temp', 'airTemp', 'humidity', 'accel', 'rumination', 'ageInDays', 'aveHerdTemp', 'THI', 'rumination_per_day']


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
from pathlib import Path

def train_catboost_timeseries(
    df: pd.DataFrame,
    feature_cols: list,
    test_size: float = 0.2,
    random_state: int = 42,
    iterations: int = 400,
    depth: int = 4,
    learning_rate: float = 0.05,
):
    """Train CatBoost with time-series aware splitting (by calf ID)"""
    df = df.copy()
    
 
    y = df["label"].astype(int)
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ids = df["id"]


    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(X, y, groups=ids))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Train set: {len(X_train)} samples, Sick ratio: {y_train.mean():.2%}")
    print(f"Test set: {len(X_test)} samples, Sick ratio: {y_test.mean():.2%}")

    # Train CatBoost
    model = CatBoostClassifier(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=random_state,
        verbose=100,  
        auto_class_weights="Balanced",  # Handle class imbalance
    )

    model.fit(X_train, y_train, eval_set=(X_test, y_test))

    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    print("CLASSIFICATION REPORT")
    print(classification_report(y_test, y_pred, digits=3))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

    return {
        "model": model,
        "X_test": X_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "feature_cols": feature_cols,
    }

In [33]:
feature_cols = [c for c in df.columns 
                if c not in ['label', 'id', 'datetime'] 
                and pd.api.types.is_numeric_dtype(df[c])]

print(f"Total features: {len(feature_cols)}")

Total features: 29


In [34]:
results = train_catboost_timeseries(
    df=df,
    feature_cols=feature_cols,
    test_size=0.2,
    random_state=42,
    iterations=400,
    depth=4,
    learning_rate=0.05
)

Train set: 698611 samples, Sick ratio: 19.17%
Test set: 182748 samples, Sick ratio: 21.39%
0:	test: 0.9476056	best: 0.9476056 (0)	total: 202ms	remaining: 1m 20s
100:	test: 0.9861020	best: 0.9861020 (100)	total: 4.7s	remaining: 13.9s
200:	test: 0.9874010	best: 0.9874048 (199)	total: 8.9s	remaining: 8.81s
300:	test: 0.9879440	best: 0.9879460 (299)	total: 13.1s	remaining: 4.29s
399:	test: 0.9883445	best: 0.9883599 (393)	total: 17.1s	remaining: 0us

bestTest = 0.9883599475
bestIteration = 393

Shrink model to first 394 iterations.

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0      0.983     0.940     0.961    143661
           1      0.810     0.941     0.870     39087

    accuracy                          0.940    182748
   macro avg      0.896     0.940     0.916    182748
weighted avg      0.946     0.940     0.942    182748


Confusion Matrix:
[[135017   8644]
 [  2316  36771]]

ROC-AUC Score: 0.9884


In [37]:
import joblib

In [38]:
joblib.dump(results['model'], 'models/catboost_fever_model_6h.joblib')
print("\nModel saved as 'catboost_fever_model.joblib'")


Model saved as 'catboost_fever_model.joblib'


In [43]:
with open('CatBoost_deployment_artifacts/feature_names.txt', 'w') as f:
    for feature in feature_cols:
        f.write(f"{feature}\n")
print(" Saved: feature_names.txt")

 Saved: feature_names.txt


In [ ]:
new_data = pd.read_csv("Data/Test/Test-animal-ids-783-new-farm-66.csv")
new_data.head()

,datetime,temp,airTemp,humidity,accel,rumination,ageInDays,aveHerdTemp,id,THI,rumination_per_day
0,2025-10-27 0:13,38.9,7.5,65.8,0.18,1,272,39.025,783,47.86,585
1,2025-10-27 0:28,38.9,7.5,65.8,0.19,0,272,38.225,783,47.86,585
2,2025-10-27 0:43,38.8,7.2,63.5,0.16,1,272,38.850,783,47.59,585
3,2025-10-27 0:58,39.0,7.2,64.9,0.07,0,272,38.950,783,47.49,585
4,2025-10-27 1:13,39.2,7.2,66.4,0.08,0,272,38.175,783,47.38,585


In [ ]:
model_config = {
    'model': results['model'],
    'feature_cols': results['feature_cols'],
    'thresholds': {
        'default': 0.500,
        'balanced': 0.651,
        'conservative': 0.700
    }
}

joblib.dump(model_config, 'models/catboost_model_complete.joblib')



['models/catboost_model_complete.joblib']

In [ ]:
model_config = joblib.load('models/catboost_model_complete.joblib')
model = model_config['model']
feature_cols = model_config['feature_cols']

## Test CatBoost Model

In [ ]:
def add_rolling_features(df, window_hours=6):
    """
    Add rolling window features to the dataframe.
    """
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values(['id', 'datetime'])
    
    numeric_cols = ['temp', 'airTemp', 'humidity', 'accel', 'rumination', 
                    'ageInDays', 'aveHerdTemp', 'THI', 'rumination_per_day']
    
    if 'id' in df.columns and pd.api.types.is_numeric_dtype(df['id']):
        numeric_cols.append('id')
    
    rolling_features = []
    
    for calf_id, group in df.groupby('id'):
        group = group.set_index('datetime')
        
        for col in numeric_cols:
            if col in group.columns:
                group[f'{col}_{window_hours}h_mean'] = group[col].rolling(
                    window=f'{window_hours}h', min_periods=1
                ).mean()
                
                group[f'{col}_{window_hours}h_std'] = group[col].rolling(
                    window=f'{window_hours}h', min_periods=1
                ).std()
        
        rolling_features.append(group.reset_index())
    
    df_with_features = pd.concat(rolling_features, ignore_index=True)
    rolling_cols = [col for col in df_with_features.columns if f'{window_hours}h' in col]
    df_with_features[rolling_cols] = df_with_features[rolling_cols].fillna(0)
    
    return df_with_features


def predict_on_new_data(model, new_df, feature_cols):
    """
    Predict illness with RECENT window confirmation (last 1.5 hours, not full 6 hours)
    """
    df_processed = add_rolling_features(new_df.copy(), window_hours=6)

    # Prepare feature matrix
    X = df_processed[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y_proba = model.predict_proba(X)[:, 1]
    threshold = 0.5
    y_pred = (y_proba >= threshold).astype(int)

    df_processed['sick_probability'] = y_proba
    df_processed['label'] = pd.Series(y_pred).map({0: 'healthy', 1: 'sick'}).values

    confirmed_list = []

    for calf_id, group in df_processed.groupby('id'):
        group = group.sort_values('datetime').copy()
        group['datetime'] = pd.to_datetime(group['datetime'])
        group = group.set_index('datetime')

        # CHANGED: Rolling window of 1.5 HOURS (recent status, not historical average)
        sick_count_recent = (group['label'] == 'sick').rolling('1.5h').sum()
        total_count_recent = group['label'].rolling('1.5h').count()

        # Confirmation: >50% sick in RECENT 1.5-hour window
        confirmed_raw = (sick_count_recent / total_count_recent) > 0.5
        
        # Show confirmation every 1.5 hours
        confirmed_output = []
        start_time = group.index[0]
        last_checkpoint = -999

        for idx_pos, idx in enumerate(group.index):
            hours_elapsed = (idx - start_time).total_seconds() / 3600
            
            checkpoint_interval = 1.5
            current_checkpoint = int(hours_elapsed / checkpoint_interval)
            
            # Show confirmation at checkpoints
            if idx_pos >= 5 and current_checkpoint > last_checkpoint:
                last_checkpoint = current_checkpoint
                confirmed_output.append("sick" if confirmed_raw.iloc[idx_pos] else "healthy")
            else:
                confirmed_output.append("")

        group['confirmed_illness'] = confirmed_output
        confirmed_list.append(group.reset_index())

    df_processed = pd.concat(confirmed_list, ignore_index=True)

    return df_processed[
        ['datetime', 'id', 'temp', 'airTemp', 'humidity', 'accel',
         'rumination', 'ageInDays', 'aveHerdTemp', 'THI',
         'rumination_per_day', 'label', 'confirmed_illness', 'sick_probability']
    ]


predictions = predict_on_new_data(model, new_data, feature_cols)


PREDICTION RESULTS - First 30 Rows
           datetime  id  temp  airTemp  humidity  accel  rumination  ageInDays  aveHerdTemp   THI  rumination_per_day   label confirmed_illness  sick_probability
2025-10-27 00:13:00 783  38.9      7.5      65.8   0.18           1        272       39.025 47.86                 585 healthy                        5.849007e-10
2025-10-27 00:28:00 783  38.9      7.5      65.8   0.19           0        272       38.225 47.86                 585 healthy                        7.945058e-10
2025-10-27 00:43:00 783  38.8      7.2      63.5   0.16           1        272       38.850 47.59                 585 healthy                        6.007870e-10
2025-10-27 00:58:00 783  39.0      7.2      64.9   0.07           0        272       38.950 47.49                 585 healthy                        1.251473e-09
2025-10-27 01:13:00 783  39.2      7.2      66.4   0.08           0        272       38.175 47.38                 585 healthy                        1.3104

In [ ]:
print("SUMMARY STATISTICS")
print(f"Total observations: {len(predictions)}")
print(f"Total sick predictions: {(predictions['label'] == 'sick').sum()}")
print(f"Total healthy predictions: {(predictions['label'] == 'healthy').sum()}")
print(f"\nTotal confirmation checkpoints: {len(confirmed_rows)}")
print(f"Confirmed sick checkpoints: {(confirmed_rows['confirmed_illness'] == 'sick').sum()}")
print(f"Confirmed healthy checkpoints: {(confirmed_rows['confirmed_illness'] == 'healthy').sum()}")


print("CONFIRMED SICK OBSERVATIONS")
confirmed_sick = predictions[predictions['confirmed_illness'] == 'sick']
print(f"Total confirmed sick observations: {len(confirmed_sick)}")
print(f"Unique calves with confirmed illness: {confirmed_sick['id'].nunique()}")

if len(confirmed_sick) > 0:
    print(f"\nCalf IDs with confirmed illness:")
    for calf_id in confirmed_sick['id'].unique():
        calf_data = confirmed_sick[confirmed_sick['id'] == calf_id]
        print(f"  - Calf {calf_id}: {len(calf_data)} confirmed sick checkpoint(s)")
        print(f"    First detection: {calf_data['datetime'].min()}")
        print(f"    Latest detection: {calf_data['datetime'].max()}")


SUMMARY STATISTICS
Total observations: 2266
Total sick predictions: 580
Total healthy predictions: 1686

Total confirmation checkpoints: 379
Confirmed sick checkpoints: 95
Confirmed healthy checkpoints: 284

CONFIRMED SICK OBSERVATIONS
Total confirmed sick observations: 95
Unique calves with confirmed illness: 1

Calf IDs with confirmed illness:
  - Calf 783: 95 confirmed sick checkpoint(s)
    First detection: 2025-10-28 04:43:00
    Latest detection: 2025-11-17 15:14:00


In [ ]:

predictions.to_csv('calf_illness_predictions.csv', index=False)